# PDEFlow · train a shared heat / wave model

A recoverable three-stage workflow for the **Operator-Aware physical-state v2 model (initial u, wave initial v, typed positive D or c)**.
1. A1 learns complete physical states. 2. A2 learns graph-conditioned direct-time evolution with the state encoder frozen. 3. B freezes both and learns joint generation from typed sparse measurements.

**Large** matches the requested 65,536 training solutions, width768/384 TOTAL tokens/depth16/heads12 and 100,000 updates per stage. It is a supported configuration, **not a demonstrated optimum or a GPU-fit promise**. Extra large is configurable. Start with the real profile; it never changes capacity silently.

Use a GPU runtime (Runtime → Change runtime type → GPU). GPU availability varies. CPU is restricted to explicit Pipeline checks. Actual Colab/CUDA execution must be verified on your allocated hardware. Old checkpoints remain untouched but are incompatible; the old notebook is historical and excluded from new source packages.

Known D/c are randomly disclosed. Unknown initial u, initial v (wave only), and typed scalar are generated jointly; no whole-trajectory latent exists. No complete initial field is required at inference. Region-balanced field loss alpha=.5 is experimental; scalar reconstruction and latent velocity have separate losses.

Batch workflow after setup: **Configure (cell3) → optional selected-stage Benchmark (cell4) → Select/apply to the real Session (cell5) → review authoritative settings → Train (cell6)**. Applying is explicit, saves old/new settings and invalidates prior resource estimates. After selection, rerun only Apply and then Train (plus preflight when required); no setup/data rerun is needed.


In [ ]:
#@title 1 · Set up project (archive or private/public checkout)
from pathlib import Path
import os, sys, subprocess, zipfile, hashlib, json, tempfile, urllib.parse
REPOSITORY_URL = "" #@param {type:"string"}
REVISION = "" #@param {type:"string"}
GITHUB_SECRET_NAME = "" #@param {type:"string"}
SOURCE_ARCHIVE = "" #@param {type:"string"}
CACHE_ROOT = Path("/content/pdeflow-source") # disposable; configurable
INSTALL_PROJECT = True #@param {type:"boolean"}
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
if not REPOSITORY_URL and not SOURCE_ARCHIVE:
    from google.colab import files
    uploaded = files.upload()  # upload the source ZIP, not model weights
    if len(uploaded) != 1: raise ValueError("Upload exactly one source ZIP")
    SOURCE_ARCHIVE = next(iter(uploaded))
if REPOSITORY_URL:
    parsed = urllib.parse.urlsplit(REPOSITORY_URL)
    if parsed.username or parsed.password or parsed.query or parsed.fragment: raise ValueError("Do not put tokens in a URL; use a runtime secret name")
    if not REVISION: raise ValueError("Set an exact checkout revision for reproducible training")
    source_key = hashlib.sha256((REPOSITORY_URL + REVISION).encode()).hexdigest()
    PACKAGE_ROOT = CACHE_ROOT / source_key[:16]
    if not PACKAGE_ROOT.exists():
        env = os.environ.copy()
        with tempfile.TemporaryDirectory() as auth:
            if GITHUB_SECRET_NAME:
                from google.colab import userdata
                token = userdata.get(GITHUB_SECRET_NAME)
                ask = Path(auth) / "askpass.py"
                ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["PDEFLOW_GIT_TOKEN"])\n')
                ask.chmod(0o700); env.update(GIT_ASKPASS=str(ask), PDEFLOW_GIT_TOKEN=token, GIT_TERMINAL_PROMPT="0")
            for cmd in (["git", "clone", "--no-checkout", REPOSITORY_URL, str(PACKAGE_ROOT)], ["git", "-C", str(PACKAGE_ROOT), "checkout", "--detach", REVISION]):
                result = subprocess.run(cmd, env=env, capture_output=True, text=True)
                if result.returncode: raise RuntimeError("Checkout failed; check revision/access. Credentials are not printed.")
    revision = subprocess.check_output(["git", "-C", str(PACKAGE_ROOT), "rev-parse", "HEAD"], text=True).strip()
    print("Source revision:", revision)
else:
    archive = Path(SOURCE_ARCHIVE).expanduser().resolve()
    source_key = hashlib.sha256(archive.read_bytes()).hexdigest()
    extracted = CACHE_ROOT / source_key[:16]
    if not extracted.exists():
        with zipfile.ZipFile(archive) as z:
            if sum(i.file_size for i in z.infolist()) > 512 * 1024**2: raise ValueError("Source ZIP exceeds 512 MiB unpacked")
            for i in z.infolist():
                p = Path(i.filename)
                if p.is_absolute() or ".." in p.parts or (i.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError("Unsafe archive member")
            z.extractall(extracted)
    roots = [p.parent for p in extracted.rglob("pyproject.toml") if (p.parent / "src/pdeflow").is_dir()]
    if len(roots) != 1: raise ValueError("ZIP must contain exactly one PDEFlow source root")
    PACKAGE_ROOT = roots[0]
    # Rerun safety: verify cached members rather than trusting a stale extraction.
    with zipfile.ZipFile(archive) as z:
        for i in z.infolist():
            member = Path(i.filename)
            if member.is_absolute() or ".." in member.parts or (i.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError("Unsafe archive member")
            if i.is_dir(): continue
            cached = extracted / member
            if cached.is_symlink() or not cached.resolve().is_relative_to(extracted.resolve()) or not cached.is_file() or hashlib.sha256(cached.read_bytes()).digest() != hashlib.sha256(z.read(i)).digest():
                raise ValueError("Cached source changed; choose a fresh CACHE_ROOT. Existing files are not overwritten.")
    print("Source archive SHA256:", source_key)
subprocess.run([sys.executable, str(PACKAGE_ROOT / "scripts/colab_setup.py"), *([] if INSTALL_PROJECT else ["--check-only"])], check=True)
# Editable .pth files installed by subprocess pip are not reloaded in this kernel.
if "pdeflow" in sys.modules and not Path(sys.modules["pdeflow"].__file__).resolve().is_relative_to(PACKAGE_ROOT.resolve()):
    raise RuntimeError("Source changed in this running kernel. Restart, then select the intended source again.")
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
import importlib
importlib.invalidate_caches()
# --no-deps setup retains the runtime's compatible PyTorch/CUDA stack.
# If verification fails, select a supported Python3.11/3.12 runtime (e.g. 2026.07).
# No automatic PyTorch replacement or hidden runtime restart occurs.


## 2 · Persistent storage

Train and checkpoint on fast local disk. Sync verified, content-addressed snapshots to Drive or another configured filesystem. Recovery ignores incomplete uploads and verifies hashes. **Updates after the last successful sync may be lost if the runtime disappears.** Drive mount requires your own interactive authorization; it is optional.


In [ ]:
#@title 2 · Connect storage
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_MOUNT = "/content/drive" #@param {type:"string"}
PERSISTENT_ROOT = "/content/drive/MyDrive/PDEFlow-training" #@param {type:"string"}
LOCAL_ROOT = "/content/pdeflow-training" #@param {type:"string"}
SYNC_SECONDS = 300 #@param {type:"integer"}
KEEP_SNAPSHOTS = 3 #@param {type:"integer"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
Path(PERSISTENT_ROOT).mkdir(parents=True, exist_ok=True)
Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
from pdeflow.colab import discover
print("Complete persistent runs:", discover(PERSISTENT_ROOT))


## 3 · Choose the experiment

Large: **65,536 / 2,048 / 2,048** solutions, 64² reference grid, continuous physical times over [0,1]; **100k A1 +100k A2 +100k flow updates**, width768, tokens384, layers16, heads12, encoder1024, query512, effective batch8, Adam LR .0003, checkpoint500. D=.03–.12, 1–128 observations, sigma0–.05, zero-observation probability0. Gate ratio1 is a minimal baseline check, not good accuracy.

XLarge starts at width1024/512 TOTAL tokens/depth24/heads16 (larger, unprofiled), 131,072 examples and 200k updates per stage. All settings remain editable in `ADVANCED`. No automatic shrinking. Microbatch × accumulation supports a fixed effective batch; a last smaller microbatch is weighted correctly. Examples are generated deterministically on demand, not put on GPU together. Validation and test solutions have separate seed identities; test remains untouched.

T4 precision: auto chooses FP16 with GradScaler; BF16 emulation is not native support. Preflight uses two warmup and eight measured updates per stage. Optional source comparison in cell4 runs the identical resolved workload against separate before/after source trees. Keep RUN_TRAINING=False during this investigation. See `docs/training-performance.md`.

Actual A1/A2/default-B neural microbatches are now supported. Cell4 offers optional selected-stage efficiency or effective-batch learning experiments. The following apply cell updates the actual Session. If comparing all three, effective batch must be at least32 and is held identical across variants; this is a separate experiment when original training batch is8. Set RUN_PREFLIGHT=False and RUN_SOURCE_COMPARISON=False to run only this sweep. See docs/batched-training.md.


In [ ]:
#@title 3 · Training size and settings
from pdeflow.colab import training_preset as preset, Session
RUN_NAME = "heat-large-01" #@param {type:"string"}
PRESET = "large" #@param ["smoke", "xsmall", "small", "medium", "large", "xlarge", "custom"]
FAMILIES = "joint" #@param ["joint", "heat", "wave"]
STAGES = "both" #@param ["both", "representation", "evolution", "flow"]
DEVICE = "cuda" #@param ["cuda", "mps", "cpu"]
PRECISION = "auto" #@param ["auto", "bf16", "fp16", "fp32"]
RESUME_MODE = "new" #@param ["new", "resume", "extend"]
REPRESENTATION_RUN = "" #@param {type:"string"}
GATE_RATIO = 1.0 #@param {type:"number"}
OVERRIDE_FAILED_GATE = False #@param {type:"boolean"}
ALLOW_CPU_PIPELINE_CHECK = False #@param {type:"boolean"}
# Structured advanced overrides: every CoordinateConfig field is supported.
# Example: {"training_options":{"microbatch":2,"accumulation":4,
#   "activation_checkpointing":True}, "flow_steps":150000}
# An update means one optimizer step. Extend keeps the original scheduler horizon
# and holds its terminal LR beyond that horizon; it never silently resets it.
# Continuous physical time is dimensionless, independent of display frames.
# A1 initial-state oversampling +32 jittered phase strata; A2 continuous transition pairs/triples.
# observation_placement: random/even/heat probabilities.
FIELD_LOSS = "region_balanced_mse" #@param ["region_balanced_mse", "mse", "weighted_mse", "mae"]
FEATURE_BALANCE = 0.5 #@param {type:"number"}
# Advanced field_loss holds thresholds, area/smoothing, analysis resolution and query budgets.
# Batch controls: examples per update, examples per neural pass, and explicit
# per-stage overrides. Accumulation is derived, including weighted remainders.
EFFECTIVE_BATCH = 8 #@param {type:"integer"}
GPU_MICROBATCH = "Auto" #@param ["Auto", "Manual"]
MANUAL_GPU_MICROBATCH = 8 #@param {type:"integer"}
A1_GPU_MICROBATCH = "inherit" #@param {type:"string"}
A2_GPU_MICROBATCH = "inherit" #@param {type:"string"}
B_GPU_MICROBATCH = "inherit" #@param {type:"string"}
BATCH_BUDGET_POLICY = "keep_updates" #@param ["keep_updates", "keep_examples"]
APPLY_BATCH_CHANGES_ON_RESUME = False #@param {type:"boolean"}
ADVANCED = {}
ADVANCED["field_loss"] = {"type": FIELD_LOSS, "alpha": FEATURE_BALANCE, **ADVANCED.get("field_loss", {})}
ADVANCED["family_weights"] = {"heat":.5,"wave":.5} if FAMILIES == "joint" else {FAMILIES:1.}
config = preset(PRESET, DEVICE, PRECISION, ADVANCED)
session = Session(config, RUN_NAME, LOCAL_ROOT, PERSISTENT_ROOT, ALLOW_CPU_PIPELINE_CHECK, SYNC_SECONDS, KEEP_SNAPSHOTS)
if RESUME_MODE != "new" and not (session.root / "session.json").exists():
    session.restore()
from pdeflow.runtime import write_json, read_json
if RESUME_MODE != "new":
    from dataclasses import replace
    recorded = read_json(session.root / "session.json")["config"]
    config = replace(config, field_loss=recorded.get("field_loss", {})).validate()
    session.c = config
    print("Resuming recorded field loss:", config.field_loss or "historical ordinary MSE")
write_json(session.root / "source-input.json", {"archive_or_checkout_sha256": source_key, "revision": revision if REPOSITORY_URL else None})
from pdeflow.colab import apply_training_settings
if RESUME_MODE != "new" and not APPLY_BATCH_CHANGES_ON_RESUME:
    # Resume retains recorded batch and budgets unless continuation is explicit.
    from dataclasses import replace
    from pdeflow.coordinate.contracts import CoordinateConfig
    prior = CoordinateConfig(**recorded).validate()
    config = replace(config, batch_size=prior.batch_size,
        ae_steps=prior.ae_steps, evolution_steps=prior.evolution_steps, flow_steps=prior.flow_steps,
        training_options=prior.training_options).validate()
    session = Session(config, RUN_NAME, LOCAL_ROOT, PERSISTENT_ROOT, ALLOW_CPU_PIPELINE_CHECK, SYNC_SECONDS, KEEP_SNAPSHOTS)
    print("Retaining recorded batch settings for resume. Choose explicit continuation to change them.")
else:
    def batch_value(value): return "auto" if value.lower() == "auto" else int(value)
    stage_batches = {s: {"microbatch": batch_value(v)} for s,v in
        [("autoencoder",A1_GPU_MICROBATCH),("evolution",A2_GPU_MICROBATCH),("flow",B_GPU_MICROBATCH)]
        if v.lower() != "inherit"}
    session = apply_training_settings(session, {"effective_batch": EFFECTIVE_BATCH,
        "microbatch": "auto" if GPU_MICROBATCH == "Auto" else MANUAL_GPU_MICROBATCH,
        "stages": stage_batches}, BATCH_BUDGET_POLICY,
        continuation=RESUME_MODE != "new")
    config = session.c
print("Authoritative batch plan:", json.dumps(session.training_summary(), indent=2))
info = session.inspect()
print("GPU:", info["hardware"]["gpu"])
print("Host RAM / available RAM / disk free:", info["hardware"]["host_ram_bytes"], info["hardware"]["available_host_ram_bytes"], info["hardware"]["local_disk_free_bytes"])
print("PyTorch / CUDA:", info["hardware"]["torch"], info["hardware"]["cuda_runtime"])
print("Stage parameters:", info["configuration"]["parameters"])
print("Effective batch / microbatch / accumulation:", info["configuration"]["effective_batch"], info["configuration"]["microbatch"], info["configuration"]["accumulation"])
print("Hypothetical full float32 dataset bytes (not allocated):", info["configuration"]["hypothetical_materialized_float32_bytes"])
print(info["configuration"]["estimates"])

print("Data / observations / budgets / optimizer starting points:")
print(json.dumps(config.to_dict(), indent=2))


In [ ]:
#@title 4 · Optional batch experiments: estimate, launch, inspect/cancel
# Benchmarks temporary configurations. Review results, then apply a selection in
# the next cell to update the real training Session. Training remains off.
from pdeflow import batch_experiments
from pdeflow.runtime import read_json
RUN_PREFLIGHT = False #@param {type:"boolean"}
RUN_BATCH_EXPERIMENT = False #@param {type:"boolean"}
EXPERIMENT_MODE = "efficiency" #@param ["efficiency", "learning"]
BENCHMARK_STAGES = "autoencoder" #@param {type:"string"}
CANDIDATE_MODE = "Auto" #@param ["Auto", "Editable"]
MICROBATCH_CANDIDATES = "8,16,32,64,128,256,512,1024" #@param {type:"string"}
FIXED_EFFECTIVE_BATCH = 8 #@param {type:"integer"}
LEARNING_EFFECTIVE_BATCHES = "8,16" #@param {type:"string"}
LEARNING_BUDGET_POLICY = "examples" #@param ["examples", "wall", "updates"]
LEARNING_BUDGET = 256 #@param {type:"number"}
EXPERIMENT_SECONDS_CAP = 120 #@param {type:"integer"}
RUN_IN_BACKGROUND = True #@param {type:"boolean"}
CANCEL_BATCH_EXPERIMENT = False #@param {type:"boolean"}
if RUN_PREFLIGHT:
    prof = session.preflight(updates=10, warmup=2)
    print(prof)
# Candidate/stage/cache subsets are explicit; Auto tests at most3 increasing sizes
# and stops larger sizes after OOM/headroom. Size1024 is not a fit promise.
experiment_options = dict(mode=EXPERIMENT_MODE,
    stages=[x.strip() for x in BENCHMARK_STAGES.split(",") if x.strip()],
    candidates="auto" if CANDIDATE_MODE == "Auto" else [int(x) for x in MICROBATCH_CANDIDATES.split(",")],
    effective_batch=FIXED_EFFECTIVE_BATCH,
    effective_batches=[int(x) for x in LEARNING_EFFECTIVE_BATCHES.split(",")],
    budget_policy=LEARNING_BUDGET_POLICY, budget=LEARNING_BUDGET,
    warmup=2, measured=5, max_seconds=EXPERIMENT_SECONDS_CAP)
print("Estimated experiment:", session.plan_batch_experiment(**experiment_options)["estimate"])
if RUN_BATCH_EXPERIMENT and not CANCEL_BATCH_EXPERIMENT:
    batch_result = session.benchmark_batch_sizes(background=RUN_IN_BACKGROUND, **experiment_options)
    BATCH_EXPERIMENT_ID = batch_result["id"]
    print("Benchmark launched; not applied:", BATCH_EXPERIMENT_ID)
if "BATCH_EXPERIMENT_ID" in globals():
    if CANCEL_BATCH_EXPERIMENT: session.stop_batch_experiment(BATCH_EXPERIMENT_ID)
    batch_result = session.batch_experiment(BATCH_EXPERIMENT_ID)
    print("State:", batch_result["state"], "· benchmark result, not applied")
    print("Stage | microbatch | accumulation | effective | examples/s | median s | peak bytes | skips | cache | status")
    for row in batch_result["rows"]:
        print(*[row.get(k,"—") for k in ["stage","microbatch","accumulation","effective_batch","examples_per_second","median_seconds","peak_allocated","skipped_updates","cache","status"]],sep=" | ")
    print("Recommendations:", batch_result["recommendation"])
    print("Rerun this cell with RUN_BATCH_EXPERIMENT=False to inspect; use cancellation checkbox to stop. Partial results persist.")


In [ ]:
#@title 5 · Apply selected batch settings to the real Session
# Rerun only this cell after choosing results. No setup/data cell rerun required.
APPLY_SELECTED_BATCH_SETTINGS = False #@param {type:"boolean"}
APPLY_SOURCE = "Recommended result" #@param ["Recommended result", "Selected result", "Manual settings"]
SELECTED_A1_MICROBATCH = 8 #@param {type:"integer"}
SELECTED_A2_MICROBATCH = 8 #@param {type:"integer"}
SELECTED_B_MICROBATCH = 8 #@param {type:"integer"}
APPLIED_EFFECTIVE_BATCH = 8 #@param {type:"integer"}
APPLIED_GLOBAL_MICROBATCH = "auto" #@param {type:"string"}
APPLY_BUDGET_POLICY = "keep_updates" #@param ["keep_updates", "keep_examples"]
EXPLICIT_BATCH_CONTINUATION = False #@param {type:"boolean"}
from pdeflow.colab import apply_training_settings

def apply_selected_batch_settings(_=None):
    global session, config
    if APPLY_SOURCE == "Manual settings":
        chosen = {"effective_batch": APPLIED_EFFECTIVE_BATCH,
            "microbatch": "auto" if APPLIED_GLOBAL_MICROBATCH.lower()=="auto" else int(APPLIED_GLOBAL_MICROBATCH),
            "stages": {"autoencoder":SELECTED_A1_MICROBATCH,
                       "evolution":SELECTED_A2_MICROBATCH,"flow":SELECTED_B_MICROBATCH}}
        session = apply_training_settings(session, chosen, APPLY_BUDGET_POLICY, EXPLICIT_BATCH_CONTINUATION)
    else:
        if "BATCH_EXPERIMENT_ID" not in globals(): raise ValueError("Run a benchmark first, or choose Manual settings")
        selected = {s:{"microbatch":m,"effective_batch":APPLIED_EFFECTIVE_BATCH} for s,m in
            [("autoencoder",SELECTED_A1_MICROBATCH),("evolution",SELECTED_A2_MICROBATCH),("flow",SELECTED_B_MICROBATCH)]
            if s in session.batch_experiment(BATCH_EXPERIMENT_ID)["plan"]["stages"]}
        session = session.apply_batch_result(BATCH_EXPERIMENT_ID,
            recommended=APPLY_SOURCE == "Recommended result", selections=selected,
            budget_policy=APPLY_BUDGET_POLICY, continuation=EXPLICIT_BATCH_CONTINUATION)
    config = session.c
    applied = read_json(session.root / "applied-settings.json")
    for stage, old in applied["application"]["before"].items():
        print(stage, "old → new:", old, "→", applied["application"]["after"][stage])
    print("APPLIED to real Session; training has not started.")
    print(json.dumps(session.training_summary(), indent=2))
    print("Previous preflight invalidated. Run cell4 preflight if required, then only the Run Training cell.")

if APPLY_SELECTED_BATCH_SETTINGS: apply_selected_batch_settings()
try:
    import ipywidgets as widgets
    from IPython.display import display
    apply_button = widgets.Button(description="Apply selected batch settings", layout=widgets.Layout(width="260px"))
    apply_button.on_click(apply_selected_batch_settings)
    display(apply_button)
except ImportError:
    print("Set APPLY_SELECTED_BATCH_SETTINGS=True and run this cell to apply. Same public API; no internal edits.")


## 6 · Start or resume

Set **RUN_TRAINING=True** to launch the selected workflow. The default False lets Run all set up and profile safely. `both` manages stage paths automatically. `representation` pauses after its diagnostics; `flow` reuses this run’s completed representation or `REPRESENTATION_RUN`.

If the representation fails its validation gate, it is saved and flow does not start. Inspect errors, then deliberately set `OVERRIDE_FAILED_GATE=True` and `RESUME_MODE="resume"` to continue. This decision is recorded. Use notebook **Interrupt execution once** or `session.stop()` for an orderly optimizer-boundary stop and final sync; a forced runtime termination can only recover the last committed snapshot.

After reconnecting: set up the **same source**, connect the same persistent root, select the same run name/configuration, choose `resume`, profile the allocated GPU, and run this one action again. Architecture/data changes are rejected; runtime/evaluation settings may change. Scientific source changes need an explicit recorded `allow_source_change` override and are not exact reproduction.


In [ ]:
#@title 6 · One action: run the selected stages
RUN_TRAINING = False #@param {type:"boolean"}
from pdeflow.notebook import run, display_status
if RUN_TRAINING:
    print("Next actual run:", json.dumps(session.training_summary(), indent=2))
    outcome = run(session, stages=STAGES, mode=RESUME_MODE,
        representation_run=REPRESENTATION_RUN, gate_ratio=GATE_RATIO,
        override_gate=OVERRIDE_FAILED_GATE)
    print("Workflow:", outcome["state"])
else:
    print("Training is off. Inspect preflight, then set RUN_TRAINING=True to start/resume.")


In [ ]:
#@title 7 · Download model for PDEFlow
DOWNLOAD_MODEL = True #@param {type:"boolean"}
EVALUATE_UNTOUCHED_TEST = False #@param {type:"boolean"}
if (session.root / "flow/resume.pt").exists():
    exported = session.export()  # also saves persistently; partial budgets labelled
    metadata = json.loads(exported.with_suffix(".json").read_text())
    print(metadata["training_status"], "· completed flow updates:", metadata["completed_updates"])
    print("SHA256:", metadata["sha256"])
    print("Export:", exported)
    if EVALUATE_UNTOUCHED_TEST:
        if metadata["training_status"] != "completed budget": raise ValueError("Finish training before the final untouched-test evaluation")
        from pdeflow.coordinate.validation import flow as evaluate_final
        evaluate_final(session.root / "flow/coordinate.pt", session.root / "final-test.json", DEVICE, split="test", subset=6)
        session.sync(True)
        print("Explicit final test saved separately. Do not use it to select training budgets.")
    if DOWNLOAD_MODEL:
        from google.colab import files
        files.download(str(exported))
else:
    print("No valid flow state yet. Complete representation/gate and start flow first.")


## Back on your Mac

1. Download `coordinate.pt` (the sidecar JSON is useful provenance; inference needs only the PT).
2. Start PDEFlow and open the Heat / Wave playground.
3. **Import model** (or Advanced → Import another model), select it, take measurements and **Run model**.

The export embeds the graph/evolution, representation decoders, flow, transforms, field schema and known-physics compatibility. No optimizer, training dataset, external AE file, Drive path or full reference is needed for inference. Partial/pipeline exports require experimental use. More parameters or query pixels do not establish accuracy or calibrated uncertainty.

Full operating/extension/storage details: `docs/colab-training.md`. Existing local commands and the legacy notebook remain available. CPU notebook-equivalent execution is verified in repository tests; actual Colab GPU AMP, fused-kernel choice, memory headroom, Drive authorization and disconnection behavior require a real Colab GPU run.
